# 01. 노드/간선/건물 데이터 로딩 + 그래프 생성

## 1) 기본 세팅

In [22]:
from pathlib import Path
import re
import math
import pandas as pd
import networkx as nx
from geopy.distance import geodesic

In [23]:
# === 파일 경로 ===
DATA_DIR = Path("../data")
PATH_NODES = DATA_DIR / "경북대 노드 정보.txt"
PATH_EDGES = DATA_DIR / "경북대 간선 정보.txt"
PATH_BUILDINGS = DATA_DIR / "쉼표 넣음.csv"

In [24]:
ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

## 2) 노드 파일

In [25]:
# === 노드 파일 로드 함수 ===
def load_nodes_txt(path):
    nodes = {} # node_id -> (lat, lng)
    pat_coord = re.compile(r"\(\s*([-\d.]+)\s*,\s*([-\d.]+)\s*\)") # (lat, lng)
    pat_id = re.compile(r"^\s*(\d+)\s*$") # node_id

    with open(path, "r", encoding="utf-8") as f:
        lines = [ln.strip() for ln in f.readlines() if ln.strip()] # 빈 줄 제거

    i = 0
    while i < len(lines): # 한 줄씩 읽기
        m1 = pat_coord.search(lines[i]) # 위경도 추출
        if not m1:
            raise ValueError(f"위경도 형식 오류: line={i+1}, text={lines[i]}")
        lat = float(m1.group(1))
        lng = float(m1.group(2))
        i += 1
        if i >= len(lines):
            raise ValueError("좌표 다음 줄에 노드 ID가 필요합니다.")
        m2 = pat_id.match(lines[i])
        if not m2:
            raise ValueError(f"노드 ID 형식 오류: line={i+1}, text={lines[i]}")
        node_id = int(m2.group(1))
        nodes[node_id] = (lat, lng)
        i += 1
    return nodes

In [26]:
# === 노드 파일 로드 테스트 ===
nodes = load_nodes_txt(PATH_NODES)
len(nodes)

330

In [27]:
list(list(nodes.items())[:3])

[(1, (35.887091, 128.6067)),
 (2, (35.892503, 128.61031)),
 (3, (35.892313, 128.61031))]

## 3) 간선 파일

In [28]:
def load_edges_txt(path):
    edges = set() # (u, v) 쌍의 집합 - 중복 제거용
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            parts = line.split()
            ids = [int(p) for p in parts]
            if len(ids) < 2:
                # 이웃이 없는 라인은 스킵
                continue
            u, neighbors = ids[0], ids[1:]
            for v in neighbors:
                if u == v: 
                    continue
                a, b = sorted((u, v)) # (작은 번호, 큰 번호)
                edges.add((a, b))  # (u, v)와 (v, u)를 같은 간선으로 처리
    return sorted(edges)

In [29]:
edges = load_edges_txt(PATH_EDGES)
len(edges), edges[:5]

(440, [(1, 113), (1, 198), (2, 3), (2, 211), (2, 216)])

## 4) 건물 CSV 로딩

In [30]:
buildings = pd.read_csv(PATH_BUILDINGS, encoding="cp949")
buildings.head()

,building_name,\tdong_num,\tarea,\theit,\t\tlat,\tlng,radius,Unnamed: 7
0,P/P공대구조실습실,\t403.구조실습실,1331.75,20.30,\t35.887395447969006\t,128.6077289,20.589055,NaN
1,경북대학교,\t24.자연과학대학,1338.60,21.60,\t35.890262189534084,\t128.60659021651662\t,20.641938,NaN
2,경북대학교\t,131.동물사육장\t,80.41,4.22,\t35.8893218932244\t,128.60674061941992\t,5.059180,NaN
3,경북대학교,\t23.사회과학대학,2586.39,19.65,35.888540802029084\t,128.61558825885922\t,28.692743,NaN
4,경북대학교,\t55.사과포장개발실\t,342.30,7.15,35.890789909613595\t,128.60918944279965\t,10.438270,NaN


In [31]:
# 모든 문자열에서 \t 제거
buildings = buildings.replace(r'\t', '', regex=True)

# 컬럼 이름에 있는 \t 제거
buildings.columns = buildings.columns.str.replace(r'\t', '', regex=True)

buildings.head()

,building_name,dong_num,area,heit,lat,lng,radius,Unnamed: 7
0,P/P공대구조실습실,403.구조실습실,1331.75,20.30,35.887395447969006,128.6077289,20.589055,NaN
1,경북대학교,24.자연과학대학,1338.60,21.60,35.890262189534084,128.60659021651662,20.641938,NaN
2,경북대학교,131.동물사육장,80.41,4.22,35.8893218932244,128.60674061941992,5.059180,NaN
3,경북대학교,23.사회과학대학,2586.39,19.65,35.888540802029084,128.61558825885922,28.692743,NaN
4,경북대학교,55.사과포장개발실,342.30,7.15,35.890789909613595,128.60918944279965,10.438270,NaN


In [32]:
buildings = buildings.rename(columns={"heit": "height"})  # 오타 통일(선호)
buildings.head()

,building_name,dong_num,area,height,lat,lng,radius,Unnamed: 7
0,P/P공대구조실습실,403.구조실습실,1331.75,20.30,35.887395447969006,128.6077289,20.589055,NaN
1,경북대학교,24.자연과학대학,1338.60,21.60,35.890262189534084,128.60659021651662,20.641938,NaN
2,경북대학교,131.동물사육장,80.41,4.22,35.8893218932244,128.60674061941992,5.059180,NaN
3,경북대학교,23.사회과학대학,2586.39,19.65,35.888540802029084,128.61558825885922,28.692743,NaN
4,경북대학교,55.사과포장개발실,342.30,7.15,35.890789909613595,128.60918944279965,10.438270,NaN


In [33]:
buildings[buildings[["lat", "lng"]].isnull().any(axis=1)]

,building_name,dong_num,area,height,lat,lng,radius,Unnamed: 7
26,경북대학교,105.온실5-1,64.64,4.16,NaN,,4.536028,NaN


In [34]:
# lat 또는 lng 값이 NaN인 행 제거
buildings = buildings.dropna(subset=["lat", "lng"]).reset_index(drop=True)

# lat 또는 lng 값이 NaN인 행이 없어야 함
buildings[buildings[["lat", "lng"]].isnull().any(axis=1)]

,building_name,dong_num,area,height,lat,lng,radius,Unnamed: 7


In [35]:
buildings["lat"] = pd.to_numeric(buildings["lat"], errors="coerce")
buildings["lng"] = pd.to_numeric(buildings["lng"], errors="coerce")

In [36]:
print("건물 수:", len(buildings))

건물 수: 116


## 5) 그래프 생성 + 간선 길이(m) 계산하기

거리 계산은 위경도 기준 geodesic(WGS84 타원체)로 미터를 구한다.

In [37]:
G = nx.Graph()

# 노드 추가 (위치 속성 포함)
for nid, (lat, lng) in nodes.items():
    G.add_node(nid, lat=lat, lng=lng) # 노드 속성

# 간선 추가 + 길이
missing_nodes = 0 # 간선 정보에 없는 노드 수 세기용
for u, v in edges: # (u, v) 쌍
    if u not in G.nodes or v not in G.nodes: # 노드 정보에 없는 노드
        missing_nodes += 1
        continue
    p_u = (G.nodes[u]["lat"], G.nodes[u]["lng"]) # (lat, lng)
    p_v = (G.nodes[v]["lat"], G.nodes[v]["lng"]) # (lat, lng)
    length_m = geodesic(p_u, p_v).meters # 두 점 사이 거리(m)
    G.add_edge(u, v, length_m=length_m) # 간선 속성

In [38]:
print(f"노드 수: {G.number_of_nodes()}")
print(f"간선 수: {G.number_of_edges()} (원본에서 누락 노드가 있어 건너뛴 간선: {missing_nodes})")

노드 수: 330
간선 수: 440 (원본에서 누락 노드가 있어 건너뛴 간선: 0)


**고립노드**: 어떤 간선(edge)에도 연결되지 않은 노드(node)

In [39]:
# 고립 노드 체크
isolated = list(nx.isolates(G))
print(f"고립 노드 수: {len(isolated)}")
if isolated:
    print("예시:", isolated[:10])

고립 노드 수: 0


## 6) 기본 검증(연결성/요약)

In [40]:
# 연결성: 가장 큰 연결 성분 크기
components = sorted(nx.connected_components(G), key=len, reverse=True)
print(f"연결 성분 개수: {len(components)}")
print(f"가장 큰 성분 노드 수: {len(components[0])}")

연결 성분 개수: 1
가장 큰 성분 노드 수: 330


In [41]:
# 거리 통계
import numpy as np
lens = [d["length_m"] for _,_,d in G.edges(data=True)]
print(f"간선 길이(m) 통계: count={len(lens)}, mean={np.mean(lens):.1f}, median={np.median(lens):.1f}, min={np.min(lens):.1f}, max={np.max(lens):.1f}")

간선 길이(m) 통계: count=440, mean=40.1, median=36.4, min=1.3, max=133.7


## 7) 저장(다음 단계에서 사용)

In [42]:
import pickle

# 그래프 저장
with open(ARTIFACTS_DIR / "graph_base.pkl", "wb") as f:
    pickle.dump(G, f)

# 건물 저장
buildings.to_parquet(ARTIFACTS_DIR / "buildings.parquet", index=False)

print("저장 완료:", ARTIFACTS_DIR)

저장 완료: ..\artifacts
